# Explainability Phase

This notebook focuses on the interpretation of trained ML models. It leverages SHAP for linear models and fallback mechanisms for Naive Bayes.

It generates both global and local explainability artifacts, visualizations, and summary reports.

In [1]:
import os
import sys
import scipy.sparse
import numpy as np
import pandas as pd
import joblib

# Add project root to path
sys.path.append(os.path.abspath(os.path.join('..')))

from src.config.settings import (
    X_TEST_PATH,
    Y_TEST_PATH,
    TFIDF_VECTORIZER_PATH,
    NB_BEST_MODEL_PATH,
    LR_BEST_MODEL_PATH,
    SVM_BEST_MODEL_PATH,
    VALID_LABELS
)
from src.config.explainability_config import ExplainabilityConfig
from src.explainability.explainer_factory import ExplainerFactory
from src.explainability.local_explanation import LocalExplanation
from src.explainability.global_explanation import GlobalExplanation
from src.explainability.text_highlighter import TextHighlighter
from src.explainability.report_generator import ReportGenerator
from src.explainability.persistence import Persistence
from src.visualization.explainability_viz import ExplainabilityVisualizer


## 1. Load Configuration and Data

In [2]:
config = ExplainabilityConfig()
viz = ExplainabilityVisualizer(config)
persistence = Persistence()
report_gen = ReportGenerator(os.path.join(config.output_directory, "explainability_summary.md"))

# Load test data
X_test = scipy.sparse.load_npz(X_TEST_PATH)
y_test = pd.read_csv(Y_TEST_PATH)
class_names = sorted(list(VALID_LABELS))

# Load TF-IDF
tfidf = joblib.load(TFIDF_VECTORIZER_PATH)
feature_names = tfidf.get_feature_names_out()


## 2. Load Models

In [3]:
models = {
    "NaiveBayes": joblib.load(NB_BEST_MODEL_PATH),
    "LogisticRegression": joblib.load(LR_BEST_MODEL_PATH),
    "SVM": joblib.load(SVM_BEST_MODEL_PATH)
}

report_data = {
    "models_loaded": list(models.keys()),
    "num_features": len(feature_names),
    "num_classes": len(class_names),
    "global_files": [],
    "local_files": [],
    "figures": [],
    "csv_files": []
}


## 3. Generate Explanations

In [4]:
# We will process one model for detailed explanation (e.g., LogisticRegression) 
# or iterate through them. Let's use LR as the primary explainer demonstration.
model_name = "LogisticRegression"
best_model = models[model_name]

# 3.1 Initialize Explainer
explainer = ExplainerFactory.get_explainer(best_model)
report_data["explainer_used"] = type(explainer).__name__

# 3.2 Calculate SHAP values for a subset to save time (e.g., first 500 instances)
subset_idx = 500
X_test_subset = X_test[:subset_idx]
shap_values = explainer.get_shap_values(X_test_subset)

npy_path = os.path.join(config.raw_directory, "shap_values.npy")
persistence.save_numpy(shap_values, npy_path)
report_data["global_files"].append(npy_path)


## 4. Global Explainability

In [5]:
global_exp = GlobalExplanation(feature_names, class_names)

# 4.1 Global Feature Importance
global_df = global_exp.get_global_importance(shap_values)
global_csv_path = os.path.join(config.output_directory, "global_feature_importance.csv")
persistence.save_csv(global_df, global_csv_path)
report_data["csv_files"].append(global_csv_path)

# 4.2 Top Features Per Class
top_class_features = global_exp.get_top_features_per_class(global_df, top_k=config.top_k_words)
class_imp_path = os.path.join(config.output_directory, "class_feature_importance.json")
persistence.save_json(top_class_features, class_imp_path)
report_data["global_files"].append(class_imp_path)


## 5. Local Explainability

In [6]:
local_exp = LocalExplanation(feature_names)
highlighter = TextHighlighter(feature_names)

# Analyze the first instance
instance_idx = 0
sample_x = X_test_subset[instance_idx]
# We need a dense sample for prediction if model expects it, but usually sparse is fine
predicted_class = best_model.predict(sample_x)[0]
pred_idx = list(best_model.classes_).index(predicted_class)

instance_shap = shap_values[instance_idx]

# 5.1 Local Explanation Details
explanation = local_exp.get_explanation(instance_shap, pred_idx, top_k=config.top_k_words)
local_json_path = os.path.join(config.output_directory, "local_explanations.json")
persistence.save_json(explanation, local_json_path)
report_data["local_files"].append(local_json_path)

# 5.2 Text Highlight Data
sample_text_tokens = ["contoh", "teks", "cyberbullying"] # In a real scenario, tokenize the original text
highlight_data = highlighter.get_highlight_data(sample_text_tokens, instance_shap, pred_idx)
highlight_csv_path = os.path.join(config.output_directory, "highlighted_words.json")
persistence.save_json(highlight_data, highlight_csv_path)
report_data["local_files"].append(highlight_csv_path)


## 6. Visualizations

In [7]:
if config.save_figures:
    viz.plot_global_feature_importance(global_df, "feature_importance")
    report_data["figures"].append("feature_importance.png")
    
    viz.plot_class_importance(top_class_features, "class_importance")
    report_data["figures"].append("class_importance.png")
    
    viz.plot_local_explanation(explanation, "local_explanation")
    report_data["figures"].append("local_explanation.png")

    # Note: SHAP summary plots require the `shap` library plotting functions directly
    # which are wrapped in the visualizer.
    X_test_dense = X_test_subset.toarray() if hasattr(X_test_subset, "toarray") else X_test_subset
    viz.plot_shap_summary(shap_values, X_test_dense, feature_names, "shap_summary")
    report_data["figures"].append("shap_summary.png")
    
    viz.plot_shap_bar(shap_values, X_test_dense, feature_names, "shap_bar")
    report_data["figures"].append("shap_bar.png")


## 7. Generate Summary Report

In [8]:
report_gen.generate_summary_report(report_data)
print("Explainability phase completed. Report generated at:", report_gen.output_path)


Explainability phase completed. Report generated at: /home/zapp/Kampus/PM/reports/explainability/explainability_summary.md
